In [15]:
import os
from pathlib import Path

import polars as pl
from sklearn.model_selection import train_test_split

In [10]:
DATASET_PATH = Path("datasets/dataset_20260706.csv")

In [11]:
DATABASE_URI = os.environ["DATABASE_URI"]

In [12]:
RANDOM_SEED = 42

# Chargement des données

In [13]:
df_dataset = pl.read_csv(DATASET_PATH)

In [14]:
df_dataset

identifiant_unique_i,identifiant_unique_j,label
str,str,bool
"""07e07779-388e-5971-947b-b999ae…","""lokki_67692fc641d8badbf051ec32""",true
"""085352c0-77c6-599d-b54a-02ef2a…","""772950c8-5202-5394-b0ae-8ebbda…",false
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5487990001""",false
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488009001""",false
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488015001""",false
…,…,…
"""refashion_TLC-REFASHION-PAV-32…","""refashion_TLC-REFASHION-PAV-34…",true
"""refashion_TLC-REFASHION-PAV-33…","""refashion_TLC-REFASHION-PAV-34…",true
"""refashion_TLC-REFASHION-PAV-33…","""refashion_TLC-REFASHION-PAV-34…",true


# Chargement des variables d'intérêt depuis la base

In [16]:
df_dataset.write_database("luis.dataset_tmp", connection=DATABASE_URI)

1000

In [34]:
sql_dataset_features = """
with actions_by_actor as (
-- D'abord on calcule les actions des parents sur la base de leurs enfants
select 
	qv.identifiant_unique as acteur_id,
	(1 = any(array_agg(qv2.action_id))) as action_reparer,
	(2 = any(array_agg(qv2.action_id))) as action_acheter,
	(3 = any(array_agg(qv2.action_id))) as action_revendre,
	(4 = any(array_agg(qv2.action_id))) as action_donner,
	(5 = any(array_agg(qv2.action_id))) as action_louer,
	(6 = any(array_agg(qv2.action_id))) as action_mettreenlocation,
	(7 = any(array_agg(qv2.action_id))) as action_emprunter,
	(8 = any(array_agg(qv2.action_id))) as action_preter,
	(9 = any(array_agg(qv2.action_id))) as action_echanger,
	(11 = any(array_agg(qv2.action_id))) as action_trier,
	(12 = any(array_agg(qv2.action_id))) as action_rapporter
from
	qfdmo_vueacteur qv
inner join qfdmo_vueacteur qv3 on
	qv.est_parent
	and qv.identifiant_unique = qv3.parent_id
	and qv3.statut <> 'SUPPRIME'
inner join qfdmo_vuepropositionservice qv2 on
	qv3.identifiant_unique = qv2.acteur_id
group by
	1
union all -- on joint les actions des enfants
select
	acteur_id,
	(1 = any(array_agg(action_id))) as action_reparer,
	(2 = any(array_agg(action_id))) as action_acheter,
	(3 = any(array_agg(action_id))) as action_revendre,
	(4 = any(array_agg(action_id))) as action_donner,
	(5 = any(array_agg(action_id))) as action_louer,
	(6 = any(array_agg(action_id))) as action_mettreenlocation,
	(7 = any(array_agg(action_id))) as action_emprunter,
	(8 = any(array_agg(action_id))) as action_preter,
	(9 = any(array_agg(action_id))) as action_echanger,
	(11 = any(array_agg(action_id))) as action_trier,
	(12 = any(array_agg(action_id))) as action_rapporter
from
	qfdmo_vuepropositionservice
group by
	acteur_id
),
-- Sélection des variables à la maille acteur avec les actions précédemment sélectionnées
features as (
select
	identifiant_unique,
	nom,
	description,
	acteur_type_id,
	adresse,
	adresse_complement,
	code_postal,
	ville,
	url,
	email,
	"location",
	telephone,
	nom_commercial,
	nom_officiel,
	siren,
	siret,
	source_id,
	naf_principal,
	horaires_osm,
	horaires_description,
	public_accueilli,
	reprise,
	exclusivite_de_reprisereparation,
	uniquement_sur_rdv,
	consignes_dacces,
	action_principale_id,
	lieu_prestation,
	latitude,
	longitude,
	code_commune_insee,
	epci_id,
	aba.*
from
	qfdmo_vueacteur qv
left join actions_by_actor aba on
	qv.identifiant_unique = aba.acteur_id)
-- Join avec les acteurs sélectionnés dans le dataset
select 
	dt.*,
	f.nom as nom_i,
	f.description as description_i,
	f.acteur_type_id as acteur_type_id_i,
	f.adresse as adresse_i,
	f.adresse_complement as adresse_complement_i,
	f.code_postal as code_postal_i,
	f.ville as ville_i,
	f.url as url_i,
	f.email as email_i,
	f.telephone as telephone_i,
	f.nom_commercial as nom_commercial_i,
	f.nom_officiel as nom_officiel_i,
	f.siren as siren_i,
	f.siret as siret_i,
	f.source_id as source_id_i,
	f.naf_principal as naf_principal_i,
	f.horaires_osm as horaires_osm_i,
	f.horaires_description as horaires_description_i,
	f.public_accueilli as public_accueilli_i,
	f.reprise as reprise_i,
	f.exclusivite_de_reprisereparation as exclusivite_de_reprisereparation_i,
	f.uniquement_sur_rdv as uniquement_sur_rdv_i,
	f.consignes_dacces as consignes_dacces_i,
	f.action_principale_id as action_principale_id_i,
	f.lieu_prestation as lieu_prestation_i,
	f.latitude as latitude_i,
	f.longitude as longitude_i,
	f.code_commune_insee as code_commune_insee_i,
	f.epci_id as epci_id_i,
	f.action_reparer as action_reparer_i,
	f.action_acheter as action_acheter_i,
	f.action_revendre as action_revendre_i,
	f.action_donner as action_donner_i,
	f.action_louer as action_louer_i,
	f.action_mettreenlocation as action_mettreenlocation_i,
	f.action_emprunter as action_emprunter_i,
	f.action_preter as action_preter_i,
	f.action_echanger as action_echanger_i,
	f.action_trier as action_trier_i,
	f.action_rapporter as action_rapporter_i,
	f2.nom as nom_j,
	f2.description as description_j,
	f2.acteur_type_id as acteur_type_id_j,
	f2.adresse as adresse_j,
	f2.adresse_complement as adresse_complement_j,
	f2.code_postal as code_postal_j,
	f2.ville as ville_j,
	f2.url as url_j,
	f2.email as email_j,
	f2.telephone as telephone_j,
	f2.nom_commercial as nom_commercial_j,
	f2.nom_officiel as nom_officiel_j,
	f2.siren as siren_j,
	f2.siret as siret_j,
	f2.source_id as source_id_j,
	f2.naf_principal as naf_principal_j,
	f2.horaires_osm as horaires_osm_j,
	f2.horaires_description as horaires_description_j,
	f2.public_accueilli as public_accueilli_j,
	f2.reprise as reprise_j,
	f2.exclusivite_de_reprisereparation as exclusivite_de_reprisereparation_j,
	f2.uniquement_sur_rdv as uniquement_sur_rdv_j,
	f2.consignes_dacces as consignes_dacces_j,
	f2.action_principale_id as action_principale_id_j,
	f2.lieu_prestation as lieu_prestation_j,
	f2.latitude as latitude_j,
	f2.longitude as longitude_j,
	f2.code_commune_insee as code_commune_insee_j,
	f2.epci_id as epci_id_j,
	f2.action_reparer as action_reparer_j,
	f2.action_acheter as action_acheter_j,
	f2.action_revendre as action_revendre_j,
	f2.action_donner as action_donner_j,
	f2.action_louer as action_louer_j,
	f2.action_mettreenlocation as action_mettreenlocation_j,
	f2.action_emprunter as action_emprunter_j,
	f2.action_preter as action_preter_j,
	f2.action_echanger as action_echanger_j,
	f2.action_trier as action_trier_j,
	f2.action_rapporter as action_rapporter_j
from
	luis.dataset_tmp dt
left join features f on
	dt.identifiant_unique_i = f.identifiant_unique
left join features f2 on
	dt.identifiant_unique_j = f2.identifiant_unique
"""

In [35]:
df_features = pl.read_database_uri(sql_dataset_features, uri=DATABASE_URI)

In [36]:
df_features.shape

(2000, 83)

In [37]:
# Assert no duplicates
assert df_features.group_by(
    ["identifiant_unique_i", "identifiant_unique_j"]
).len().select(pl.col("len")).sum().item() == len(df_features)

In [38]:
df_features.describe()

statistic,identifiant_unique_i,identifiant_unique_j,label,nom_i,description_i,acteur_type_id_i,adresse_i,adresse_complement_i,code_postal_i,ville_i,url_i,email_i,telephone_i,nom_commercial_i,nom_officiel_i,siren_i,siret_i,source_id_i,naf_principal_i,horaires_osm_i,horaires_description_i,public_accueilli_i,reprise_i,exclusivite_de_reprisereparation_i,uniquement_sur_rdv_i,consignes_dacces_i,action_principale_id_i,lieu_prestation_i,latitude_i,longitude_i,code_commune_insee_i,epci_id_i,action_reparer_i,action_acheter_i,action_revendre_i,action_donner_i,…,adresse_j,adresse_complement_j,code_postal_j,ville_j,url_j,email_j,telephone_j,nom_commercial_j,nom_officiel_j,siren_j,siret_j,source_id_j,naf_principal_j,horaires_osm_j,horaires_description_j,public_accueilli_j,reprise_j,exclusivite_de_reprisereparation_j,uniquement_sur_rdv_j,consignes_dacces_j,action_principale_id_j,lieu_prestation_j,latitude_j,longitude_j,code_commune_insee_j,epci_id_j,action_reparer_j,action_acheter_j,action_revendre_j,action_donner_j,action_louer_j,action_mettreenlocation_j,action_emprunter_j,action_preter_j,action_echanger_j,action_trier_j,action_rapporter_j
str,str,str,f64,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,…,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""2000""","""2000""",2000.0,"""2000""","""2000""",2000.0,"""2000""","""2000""","""2000""","""2000""","""2000""","""2000""","""2000""","""2000""","""2000""","""2000""","""2000""",1514.0,"""2000""","""2000""","""2000""","""2000""","""2000""",308.0,351.0,"""2000""",11.0,"""1982""",2000.0,2000.0,"""2000""",1961.0,1948.0,1948.0,1948.0,1948.0,…,"""2000""","""2000""","""2000""","""2000""","""2000""","""2000""","""2000""","""2000""","""2000""","""2000""","""2000""",1905.0,"""2000""","""2000""","""2000""","""2000""","""2000""",364.0,328.0,"""2000""",13.0,"""2000""",1999.0,1999.0,"""2000""",1931.0,1989.0,1989.0,1989.0,1989.0,1989.0,1989.0,1989.0,1989.0,1989.0,1989.0,1989.0
"""null_count""","""0""","""0""",0.0,"""0""","""0""",0.0,"""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",486.0,"""0""","""0""","""0""","""0""","""0""",1692.0,1649.0,"""0""",1989.0,"""18""",0.0,0.0,"""0""",39.0,52.0,52.0,52.0,52.0,…,"""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",95.0,"""0""","""0""","""0""","""0""","""0""",1636.0,1672.0,"""0""",1987.0,"""0""",1.0,1.0,"""0""",69.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0
"""mean""",null,null,0.5,null,null,7.213,null,null,null,null,null,null,null,null,null,null,null,128.234478,null,null,null,null,null,0.003247,0.002849,null,10.363636,null,2.331025,45.771323,null,576.751147,0.066222,0.011294,0.005647,0.047741,…,null,null,null,null,null,null,null,null,null,null,null,125.373753,null,null,null,null,null,0.0,0.033537,null,11.0,null,2.315914,45.961104,null,578.249612,0.1272,0.020111,0.004022,0.092006,0.007541,0.0,0.018602,0.0,0.006536,0.829563,0.011564
"""std""",null,null,null,null,null,2.853191,null,null,null,null,null,null,null,null,null,null,null,113.003184,null,null,null,null,null,null,null,null,2.110579,null,6.738655,4.577283,null,348.36485,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,109.665309,null,null,null,null,null,null,null,null,0.0,null,6.063069,4.099164,null,347.704953,null,null,null,null,null,null,null,null,null,null,null
"""min""","""0034570d-8761-55da-b5f8-33e809…","""772950c8-5202-5394-b0ae-8ebbda…",0.0,"""10-Déchèterie de Gradignan""","""""",1.0,"""""","""""","""""","""""","""""","""""","""""","""""","""""","""""","""""",2.0,"""""","""""","""""","""""","""""",0.0,0.0,"""""",4.0,"""""",-62.852201,-21.173677,"""""",4.0,0.0,0.0,0.0,0.0,…,"""""","""""","""""","""""","""""","""""","""""","""""","""""","""""","""""",1.0,"""""","""""","""""","""""","""""",0.0,0.0,

# Traitement des "__empty__"

In [51]:
df_features_preprocessed = df_features.with_columns(
    pl.selectors.by_dtype(pl.String)
    .str.strip_chars()
    .replace("__empty__", None)
    .replace("", None)
)

In [52]:
df_features_preprocessed.describe()

statistic,identifiant_unique_i,identifiant_unique_j,label,nom_i,description_i,acteur_type_id_i,adresse_i,adresse_complement_i,code_postal_i,ville_i,url_i,email_i,telephone_i,nom_commercial_i,nom_officiel_i,siren_i,siret_i,source_id_i,naf_principal_i,horaires_osm_i,horaires_description_i,public_accueilli_i,reprise_i,exclusivite_de_reprisereparation_i,uniquement_sur_rdv_i,consignes_dacces_i,action_principale_id_i,lieu_prestation_i,latitude_i,longitude_i,code_commune_insee_i,epci_id_i,action_reparer_i,action_acheter_i,action_revendre_i,action_donner_i,…,adresse_j,adresse_complement_j,code_postal_j,ville_j,url_j,email_j,telephone_j,nom_commercial_j,nom_officiel_j,siren_j,siret_j,source_id_j,naf_principal_j,horaires_osm_j,horaires_description_j,public_accueilli_j,reprise_j,exclusivite_de_reprisereparation_j,uniquement_sur_rdv_j,consignes_dacces_j,action_principale_id_j,lieu_prestation_j,latitude_j,longitude_j,code_commune_insee_j,epci_id_j,action_reparer_j,action_acheter_j,action_revendre_j,action_donner_j,action_louer_j,action_mettreenlocation_j,action_emprunter_j,action_preter_j,action_echanger_j,action_trier_j,action_rapporter_j
str,str,str,f64,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,…,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""2000""","""2000""",2000.0,"""2000""","""51""",2000.0,"""1967""","""119""","""1996""","""1998""","""333""","""111""","""434""","""459""","""3""","""1236""","""747""",1514.0,"""83""","""6""","""44""","""1628""","""296""",308.0,351.0,"""256""",11.0,"""651""",2000.0,2000.0,"""1963""",1961.0,1948.0,1948.0,1948.0,1948.0,…,"""1950""","""114""","""1992""","""1991""","""257""","""151""","""415""","""457""","""4""","""1089""","""797""",1905.0,"""174""","""6""","""47""","""1565""","""219""",364.0,328.0,"""287""",13.0,"""586""",1999.0,1999.0,"""1931""",1931.0,1989.0,1989.0,1989.0,1989.0,1989.0,1989.0,1989.0,1989.0,1989.0,1989.0,1989.0
"""null_count""","""0""","""0""",0.0,"""0""","""1949""",0.0,"""33""","""1881""","""4""","""2""","""1667""","""1889""","""1566""","""1541""","""1997""","""764""","""1253""",486.0,"""1917""","""1994""","""1956""","""372""","""1704""",1692.0,1649.0,"""1744""",1989.0,"""1349""",0.0,0.0,"""37""",39.0,52.0,52.0,52.0,52.0,…,"""50""","""1886""","""8""","""9""","""1743""","""1849""","""1585""","""1543""","""1996""","""911""","""1203""",95.0,"""1826""","""1994""","""1953""","""435""","""1781""",1636.0,1672.0,"""1713""",1987.0,"""1414""",1.0,1.0,"""69""",69.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0
"""mean""",null,null,0.5,null,null,7.213,null,null,null,null,null,null,null,null,null,null,null,128.234478,null,null,null,null,null,0.003247,0.002849,null,10.363636,null,2.331025,45.771323,null,576.751147,0.066222,0.011294,0.005647,0.047741,…,null,null,null,null,null,null,null,null,null,null,null,125.373753,null,null,null,null,null,0.0,0.033537,null,11.0,null,2.315914,45.961104,null,578.249612,0.1272,0.020111,0.004022,0.092006,0.007541,0.0,0.018602,0.0,0.006536,0.829563,0.011564
"""std""",null,null,null,null,null,2.853191,null,null,null,null,null,null,null,null,null,null,null,113.003184,null,null,null,null,null,null,null,null,2.110579,null,6.738655,4.577283,null,348.36485,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,109.665309,null,null,null,null,null,null,null,null,0.0,null,6.063069,4.099164,null,347.704953,null,null,null,null,null,null,null,null,null,null,null
"""min""","""0034570d-8761-55da-b5f8-33e80997efb0""","""772950c8-5202-5394-b0ae-8ebbda9b955e""",0.0,"""10-Déchèterie de Gradignan""","""06 08 89 26 40""",1.0,"""0 Chemin du Parc""","""- LD THIZY -""","""01000""","""AGDE""","""http://cordonneriedesplatanes.fr/""","""2blocation.contact@gmail.com""","""+33240963172""","""3RD'Anjou""","""ANONYMISE POUR RAISON RGPD""","""200000586""","""200007201

# Train/test split

In [61]:
df_train, df_test = train_test_split(df_features_preprocessed, test_size=0.2)

In [62]:
df_train.group_by("label").len()

label,len
bool,u32
true,786
false,814


In [73]:
df_train

identifiant_unique_i,identifiant_unique_j,label,nom_i,description_i,acteur_type_id_i,adresse_i,adresse_complement_i,code_postal_i,ville_i,url_i,email_i,telephone_i,nom_commercial_i,nom_officiel_i,siren_i,siret_i,source_id_i,naf_principal_i,horaires_osm_i,horaires_description_i,public_accueilli_i,reprise_i,exclusivite_de_reprisereparation_i,uniquement_sur_rdv_i,consignes_dacces_i,action_principale_id_i,lieu_prestation_i,latitude_i,longitude_i,code_commune_insee_i,epci_id_i,action_reparer_i,action_acheter_i,action_revendre_i,action_donner_i,action_louer_i,…,adresse_j,adresse_complement_j,code_postal_j,ville_j,url_j,email_j,telephone_j,nom_commercial_j,nom_officiel_j,siren_j,siret_j,source_id_j,naf_principal_j,horaires_osm_j,horaires_description_j,public_accueilli_j,reprise_j,exclusivite_de_reprisereparation_j,uniquement_sur_rdv_j,consignes_dacces_j,action_principale_id_j,lieu_prestation_j,latitude_j,longitude_j,code_commune_insee_j,epci_id_j,action_reparer_j,action_acheter_j,action_revendre_j,action_donner_j,action_louer_j,action_mettreenlocation_j,action_emprunter_j,action_preter_j,action_echanger_j,action_trier_j,action_rapporter_j
str,str,bool,str,str,i32,str,str,str,str,str,str,str,str,str,str,str,i32,str,str,str,str,str,bool,bool,str,i32,str,f64,f64,str,i32,bool,bool,bool,bool,bool,…,str,str,str,str,str,str,str,str,str,str,str,i32,str,str,str,str,str,bool,bool,str,i32,str,f64,f64,str,i32,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
"""20af6bee-5b64-50de-8c00-5eca318fdd59""","""citeo_1619339002""",true,"""CC COUSERANS-PYRENEES: Point de collecte""",null,10,"""191 Lieu Dit Remillasse""",null,"""09200""","""Moulis""",null,null,null,null,null,"""200067940""",null,null,null,null,null,"""Particuliers""",null,null,null,null,null,null,1.093232,42.945893,"""09214""",423,false,false,false,false,false,…,"""123 Route du Portech""",null,"""09200""","""Moulis""",null,null,null,null,null,"""200067940""",null,87,null,null,null,"""Particuliers""",null,null,null,null,null,null,1.093232,42.945893,"""09214""",423,false,false,false,false,false,false,false,false,false,true,false
"""batribox_ENT180103MQ""","""capillum_722""",false,"""Lidl - TOURNUS Mouron""",null,4,"""avenue du Clos Mouron""",null,"""71700""","""Tournus""",null,null,null,"""Lidl""",null,null,null,99,null,null,null,"""Particuliers""","""1 pour 0""",false,false,null,null,null,4.904666,46.57173,"""71543""",524,false,false,false,false,false,…,"""2 RUE DU PORT""",null,"""64440""","""LARUNS""","""https://www.capillum.fr/capimap""",null,"""0559980961""",null,null,null,null,438,null,null,"""DU LUNDI AU JEUDI 9H00-12H00 / 13H30 - 18H00VENDREDI 9H00-18H00SAMEDI 8H00-14H00""",null,null,null,null,null,null,"""SUR_PLACE""",-0.426676,42.987431,"""64320""",1073,false,false,false,false,false,false,false,false,false,true,false
"""b398f581-f85b-4735-936e-de1a9ffa1543""","""ecodds_FD3788""",true,"""DECHETERIE DE MONSOLS CCSB - CC SAONE BEAUJOLAIS""",null,7,"""lieu dit au colombier,""","""Monsols""","""69860""","""Deux-Grosnes""",null,null,"""0474047709""","""DECHETERIE DE MONSOLS CCSB - CC SAONE BEAUJOLAIS""",null,null,null,null,null,null,null,"""Particuliers""",null,null,null,"""Un badge d'accès peut vous être demandé à l’entrée. Nous vous invitons à vous renseigner avant de vous rendre sur place.""",null,null,4.524567,46.225804,"""69135""",412,false,false,false,false,false,…,"""Le Colombier,Monsols""",null,"""69860""","""Deux Grosnes""",null,null,null,"""CC SAONE BEAUJOLAIS""",null,null,null,92,null,null,null,"""Particuliers et professionnels""","""1 pour 0""",false,null,null,null,null,4.523755,46.2262,"""69135""",412,false,false,false,false,false,false,false,false,false,true,false
"""b5bbc7a4-d445-5935-9b2f-0bb51d828004""","""ecosystem_EEE-ecosystem-C-1030641""",true,"""DÉCHÈTERIE CASTETS - SITCOM côte sud des Landes""",null,7,"""1825 ROUTE DE TALLER""",null,"""40260""","""Castets""","""https://www.sitcom40.fr/""",null,"""0558720394""","""DÉCHÈTERIE CASTETS- SITCOM côte sud des Landes"

In [63]:
df_test.group_by("label").len()

label,len
bool,u32
false,186
true,214


In [80]:
id_cols = ["identifiant_unique_i", "identifiant_unique_j"]
colnames = [
    "identifiant_unique_i",
    "identifiant_unique_j",
    "label",
    "nom_i",
    "description_i",
    "acteur_type_id_i",
    "adresse_i",
    "adresse_complement_i",
    "code_postal_i",
    "ville_i",
    "url_i",
    "email_i",
    "telephone_i",
    "nom_commercial_i",
    "nom_officiel_i",
    "siren_i",
    "siret_i",
    "source_id_i",
    "naf_principal_i",
    "horaires_osm_i",
    "horaires_description_i",
    "public_accueilli_i",
    "reprise_i",
    "exclusivite_de_reprisereparation_i",
    "uniquement_sur_rdv_i",
    "consignes_dacces_i",
    "action_principale_id_i",
    "lieu_prestation_i",
    "latitude_i",
    "longitude_i",
    "code_commune_insee_i",
    "epci_id_i",
    "action_reparer_i",
    "action_acheter_i",
    "action_revendre_i",
    "action_donner_i",
    "action_louer_i",
    "action_mettreenlocation_i",
    "action_emprunter_i",
    "action_preter_i",
    "action_echanger_i",
    "action_trier_i",
    "action_rapporter_i",
    "nom_j",
    "description_j",
    "acteur_type_id_j",
    "adresse_j",
    "adresse_complement_j",
    "code_postal_j",
    "ville_j",
    "url_j",
    "email_j",
    "telephone_j",
    "nom_commercial_j",
    "nom_officiel_j",
    "siren_j",
    "siret_j",
    "source_id_j",
    "naf_principal_j",
    "horaires_osm_j",
    "horaires_description_j",
    "public_accueilli_j",
    "reprise_j",
    "exclusivite_de_reprisereparation_j",
    "uniquement_sur_rdv_j",
    "consignes_dacces_j",
    "action_principale_id_j",
    "lieu_prestation_j",
    "latitude_j",
    "longitude_j",
    "code_commune_insee_j",
    "epci_id_j",
    "action_reparer_j",
    "action_acheter_j",
    "action_revendre_j",
    "action_donner_j",
    "action_louer_j",
    "action_mettreenlocation_j",
    "action_emprunter_j",
    "action_preter_j",
    "action_echanger_j",
    "action_trier_j",
    "action_rapporter_j",
]

In [82]:
df_features_preprocessed_split = (
    df_features_preprocessed.join(
        df_train.select(*id_cols, "label"),
        on=id_cols,
        how="left",
        validate="1:1",
        suffix="_train",
    )
    .join(
        df_test.select(*id_cols, "label"),
        on=id_cols,
        how="left",
        validate="1:1",
        suffix="_test",
    )
    .with_columns(
        pl.when(pl.col("label_train").is_not_null())
        .then(pl.lit("train"))
        .when(pl.col("label_test").is_not_null())
        .then(pl.lit("test"))
        .otherwise(pl.lit("unknown"))
        .alias("split")
    )
    .select(colnames + ["split"])
)

# Sauvegarde

In [83]:
df_features_preprocessed_split.write_parquet(
    DATASET_PATH.parent / f"features_{DATASET_PATH.stem}.parquet"
)